In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions
from selenium.webdriver.common.by import By

import time
import random
from tqdm import tqdm

from bs4 import BeautifulSoup
import lxml
import re
import json

In [3]:
mean = 2.0
std_dev = 0.5
least = 0.5
def random_wait():
    """ 按高斯分布随机等待时间
    """
    wait_time = max(least, random.gauss(mean, std_dev))
    time.sleep(wait_time)

In [4]:
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"

def setup_driver():
    """ 初始化 Chrome 的 webdriver
    Returns:
        WebDriver
    """
    options = Options()
    # 无头模式
    options.add_argument("--headless")
    # wsl 内必要选项: 禁用沙盒和共享内存
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    # 模拟真实 user-agent
    options.add_argument(f"user-agent={user_agent}")

    return webdriver.Chrome(options=options)

In [5]:
# 网易云网页端歌单只显示前 10 首解决方法
# 参考 https://www.bilibili.com/opus/680540454383517718
cookie = {
    "name": "os",
    "value": "pc"
}

In [6]:
root_url = "https://music.163.com"

In [8]:
songlist_ids = ['2562363367', '73246647', '90483498', '12556191058']
songlist_preurl = "https://music.163.com/#/playlist?id="

songlist_page_sources = []
driver = setup_driver()

driver.get(root_url)
driver.add_cookie(cookie)

try:
    for songlist_id in tqdm(songlist_ids, desc="爬取歌单进度"):
        driver.get(songlist_preurl + songlist_id)
        
        # 至多等待 10s, 等待 iframe 加载完成
        WebDriverWait(driver, 10).until(
            expected_conditions.frame_to_be_available_and_switch_to_it((By.ID, "g_iframe"))
        )
        songlist_page_sources.append(driver.page_source)
        # 实时缓存源代码文件
        with open(f"../data/page_sources_songlist/list{songlist_id}.html", "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        random_wait()
finally:
    driver.quit()
    

爬取歌单进度: 100%|██████████| 4/4 [00:21<00:00,  5.46s/it]


In [9]:
# 提取歌单页面源代码中歌曲 id, 在 “/song?id=” 之后
song_ids = set()
for page_source in songlist_page_sources:
    soup = BeautifulSoup(page_source, 'lxml')
    for tag in soup.find_all(name="a", href=re.compile(r"song\?id")):
        song_id = str(tag['href']).removeprefix("/song?id=")
        song_ids.add(song_id)

with open("../data/song_ids.txt", "w", encoding="utf-8") as f:
    for id in song_ids:
        f.write(id + "\n")

In [10]:
song_ids = list(song_ids)
len(song_ids)

2067

In [11]:
song_preurl = "https://music.163.com/#/song?id="

driver = setup_driver()

driver.get(root_url)
driver.add_cookie(cookie)

step = 2563
try:
    for [index, song_id] in enumerate(tqdm(song_ids, desc="爬取歌曲页面进度")):
        if index + 1 < step:
            continue

        driver.get(song_preurl + song_id)
        
        # 至多等待 10s, 等待 iframe 加载完成
        WebDriverWait(driver, 10).until(
            expected_conditions.frame_to_be_available_and_switch_to_it((By.ID, "g_iframe"))
        )
        # 实时缓存源代码文件
        with open(f"../data/page_sources_song/{song_id}.html", "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        random_wait()
finally:
    driver.quit()

爬取歌曲页面进度: 100%|██████████| 2067/2067 [00:00<00:00, 1703601.17it/s]


In [12]:
def extract_info(song_id: str) -> dict | None:
    """ 从该歌曲的 html 文件中提取所需歌曲信息
    Args:
        song_id(str): 歌曲 id
    Returns:
        dict: 歌曲信息
    """
    with open(f"../data/page_sources_song/{song_id}.html", "r", encoding="utf-8") as f:
        page_source = f.read()
    soup = BeautifulSoup(page_source, 'lxml')

    error = False
    song = dict()

    # 歌曲名
    tags = soup.find_all("div", class_="tit")
    if len(tags) == 1:
        song["name"] = tags[0].em.text
    else:
        error = True
    
    # 歌手
    tags = soup.find_all("a", class_="s-fc7", href=re.compile(r"artist\?id"))
    artists = []
    for tag in tags:
        artist_name = tag.text
        artist_id = tag["href"].removeprefix("/artist?id=")
        artists.append({"name": artist_name, "id": artist_id})
    if len(tags) >= 1:
        song["artist"] = artists
    else:
        error = True
    
    # 歌词
    tags = soup.find_all("div", id="lyric-content")
    if len(tags) == 1:
        lyric = tags[0].get_text(separator="\n")
        if tags[0].div != None:
            lyric = lyric + tags[0].div.get_text(separator="\n")
        song["lyric"] = lyric
    else:
        error = True

    # 歌曲封面图片
    tags = soup.find_all("div", class_="u-cover u-cover-6 f-fl")
    if len(tags) == 1:
        song["cover_img_url"] = tags[0].img["data-src"]
    else:
        error = True

    # 歌曲原始网站 url
    song["origin_url"] = song_preurl + song_id

    if error:
        return None
    return song

In [13]:
extract_info("460895")

{'name': '君が死んでも許してあげるよ',
 'artist': [{'name': 'きくお', 'id': '14640'}, {'name': '初音ミク', 'id': '159692'}],
 'lyric': '作词 : きくお\n作曲 : きくお\n最終列車の屋根に\n在终班车的车顶上\n私の欠片を忘れてった\n遗落了我的碎片\n拾い上げて焼いて食べた君を\n把它捡起来烤着吃着的你\n空から見てた\n我从天空看着\n君はいつもと同じで\n你还是一如既往\n私もいつもと変わらなくて\n我也和平常没什么两样\n居場所のない街の上を\n只是在没有容身之处的街道上空\nただ彷徨うだけだった\n彷徨着而已\nランラン…\n啦啦…\n終わらない悪夢を一緒に過ごしたいから\n因为想和你一起度过无尽的噩梦\n君を支える小さな希望の灯を消して\n把那支撑着你的微弱的希望灯火 给熄灭了\n心のスキマを迷路でいっぱいにする前に\n在心的间隙被迷宫填满之前\n私が連れてあげる\n我会把你带走的\n誰もいない星へ\n带到毫无人烟的星球上\nおいで おいで\n过来吧  过来吧\n遠い星から迎いに来るから\n我会从遥远的星球过来迎接你的\nいいよ いいよ\n没关系  没关系\n今さよならしていいよ\n现在说再见也没关系的\nおいで おいで\n过来吧  过来吧\n君が死んでも許してあげるよ\n你就算死了我也会原谅你的\nいいよ いいよ\n没关系  没关系\nそう囁いであげる\n我会这样对你轻轻耳语的\n肯定 肯定しよう\n去肯定  去肯定吧\n君の全てを\n把你的全部\n感情の谷から連れ出してあげる\n我会把你从感情之谷带出来的\n返して 返して 私の忘れ物\n还给我 还给我 把我的遗落之物\n終わらない悪夢を一緒に過ごしたいから\n因为想和你一起度过无尽的噩梦\n君を支える大きな 絶望に火を灯して\n把那支撑着你的巨大的绝望 给点燃了\n最終列車の屋根に\n在终班车的车顶上\n私の欠片を忘れたの\n遗落了我的碎片了哦\n目に沈む陰りを見逃さないで\n不要看漏了沉浸在眼里的阴影啊\nおいで おいで\n过来吧  过来吧\n遠い星が君にも見えたら\n若是你也看到了那遥远的星球的话\nいいよ いいよ\n没关系  没关系\n囁いであげるから\n我会对你轻轻耳语的\nおいで おいで\n

In [14]:
infos = []
for song_id in tqdm(song_ids, desc="提取歌曲信息进度"):
    info = extract_info(song_id)
    if info != None:
        infos.append(info)
with open("../data/song_info.json", "w", encoding="utf-8") as f:
    json.dump(infos, f, ensure_ascii=False)

# 有效歌曲数量
len(infos)

提取歌曲信息进度: 100%|██████████| 2067/2067 [00:36<00:00, 56.69it/s]


2045

In [15]:
artist_list = dict()
for song in infos:
    for artist in song["artist"]:
        artist_list[artist["id"]] = artist["name"]
# 艺术家数量
len(artist_list)

551

In [ ]:
artist_preurl = "https://music.163.com/#/artist?id="
artist_desc_preurl = "https://music.163.com/#/artist/desc?id="

driver = setup_driver()

driver.get(root_url)
driver.add_cookie(cookie)

step = 551
try:
    for [index, artist_id] in enumerate(tqdm(list(artist_list), desc="爬取艺术家页面进度")):
        if index + 1 < step:
            continue

        driver.get(artist_desc_preurl + artist_id)
        
        # 至多等待 10s, 等待 iframe 加载完成
        WebDriverWait(driver, 10).until(
            expected_conditions.frame_to_be_available_and_switch_to_it((By.ID, "g_iframe"))
        )
        # 实时缓存源代码文件
        with open(f"../data/page_sources_artist/{artist_id}.html", "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        random_wait()
finally:
    driver.quit()

爬取艺术家页面进度: 100%|██████████| 551/551 [20:16<00:00,  2.21s/it]


In [35]:
def extract_artist_info(artist_id: str) -> dict | None:
    """ 从该艺术家的 html 文件中提取所需艺术家信息
    Args:
        artist_id(str): 艺术家 id
    Returns:
        dict: 艺术家信息
    """
    with open(f"../data/page_sources_artist/{artist_id}.html", "r", encoding="utf-8") as f:
        page_source = f.read()
    soup = BeautifulSoup(page_source, 'lxml')

    artist = dict()

    tags = soup.find_all("div", class_="n-artist f-cb")
    if len(tags) == 0:
        return None
    artist["name"] = tags[0].div.h2.text
    artist["img_url"] = tags[0].img["src"].split("?param=")[0]

    tags = soup.find_all("div", class_="n-artdesc")
    if len(tags) == 1 and tags[0].find("p") and tags[0].p.find("text"):
        artist["description"] = tags[0].p.text
    else:
        artist["description"] = "空空如也"

    artist["origin_url"] = artist_preurl + artist_id

    return artist

In [37]:
infos = []
for artist_id in tqdm(list(artist_list), desc="提取艺术家信息进度"):
    info = extract_artist_info(artist_id)
    if info != None:
        infos.append(info)
with open("../data/artist_info.json", "w", encoding="utf-8") as f:
    json.dump(infos, f, ensure_ascii=False)

# 有效艺术家数量
len(infos)

提取艺术家信息进度: 100%|██████████| 551/551 [00:03<00:00, 143.48it/s]


545